Optimisation de l'entrainement pour `focus`

> ... TODO ...


In [ ]:
from retinotopy import *

In [ ]:
data_set_type = 'focus' # Select your root between : 'boxes', 'focus', 'full'
print(f'{data_set_type=}')
args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training



In [ ]:
args = Params()

for args.do_polar in [True, False]:
    print(f'{args.do_polar=}')
    dataloaders = datasets_transforms(args)
    for i_step, (images, labels) in enumerate(dataloaders['train']):
        images, labels = images.to('cpu'), labels.to('cpu')
        break
    imshow(images[:5], title='Example of cartesian images' if not(args.do_polar) else 'Example of log-polar images', fig_height=5)

# Training function

In [ ]:
def train_model(args, model, dataloaders, each_steps=64, verbose=True):
    # if torch.cuda.is_available():
    #     model = model.to(device, memory_format=torch.channels_last)
    #     torch.cuda.amp.GradScaler(enabled=True)
    # else:
    #     model = model.to(device)
    model = model.to(device)
    # retraining the full model
    for param in model.parameters():
        param.requires_grad = True        

    if args.beta2 > 0.: 
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(1-args.momentum, 1-args.beta2)) 
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=1-args.momentum) # to set training variables

    df_train = pd.DataFrame([], columns=['epoch', 'i_image', 'total_image', 'avg_loss', 'avg_acc', 'avg_loss_val', 'avg_acc_val', 'time']) 

    criterion = nn.CrossEntropyLoss() #binary_cross_entropy_with_logits
    total_image = 0
    since = time.time()

    n_train = len(dataloaders['train'].dataset)
    n_train_stop = args.n_train_stop
    if n_train_stop==0: n_train_stop = n_train

    for i_epoch in range(args.num_epochs):
        i_image = 0
        for i_step, (images, labels) in enumerate(dataloaders['train']):
            images, labels = images.to(device), labels.to(device)
            total_image += len(images)
            i_image += len(images)
            if i_image > n_train_stop: break

            optimizer.zero_grad()

            outputs = model(images)
             
            loss = criterion(outputs, labels)            
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs.data, dim=1)

            avg_loss = loss.item() * images.size(0)
            avg_acc = torch.mean((preds == labels.data)*1.).cpu().item()

            if (i_step % (max(n_train_stop//args.batch_size//each_steps, 1))==0) or (i_step == n_train_stop-1):
                with torch.no_grad():
                    loss_val = 0
                    acc_val = 0
                    model = model.eval()
                    n_val = len(dataloaders['val'])
                    for _, (images, labels) in enumerate(dataloaders['val']):
                        images, labels = images.to(device), labels.to(device)

                        outputs = model(images)

                        loss = criterion(outputs, labels)

                        loss_val += loss.item() * images.size(0)

                        _, preds = torch.max(outputs.data, dim=1)
                        acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

                    avg_loss_val = loss_val / n_val
                    avg_acc_val = acc_val / n_val

                    df_train.loc[len(df_train)] = {'epoch': i_epoch, 'i_image':i_image, 'total_image':total_image, 'avg_loss':avg_loss, 'avg_acc':avg_acc, 'avg_loss_val':avg_loss_val, 'avg_acc_val':avg_acc_val, 'time':time.time() - since}
                    if verbose:  print(f"Epoch {i_epoch}, i_image {i_image} : train= loss: {avg_loss:.4f} / acc : {avg_acc:.4f} - val= loss : {avg_loss_val:.4f} / acc : {avg_acc_val:.4f} / time:{time.time() - since:.1f}")

    if torch.cuda.is_available(): torch.cuda.empty_cache()        
    return model, df_train

# Training on centered images using the bounding boxes

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_width/phi), subplotpars=subplotpars)


# Training and saving the networks
for model_name in  ['resnet101']: #'resnet50']: #, 
    for do_polar, color in zip([False, True], ['b', 'r']):

        filename = f'{data_cache}/{datetag}_{model_name}_{do_polar=}'
        json_filename = filename + '.json'
        model_filename = filename + '.pt'


        if os.path.isfile(model_filename):
            print(f"Load pre-trained resnet {model_filename}")
            df_train = pd.read_json(json_filename, orient='index')
            print(f"{model_filename}: accuracy = {df_train['avg_acc_val'][-5:].mean():.3f}")

        elif os.path.isfile(model_filename + '.lock'):
            # we want to have a file but it's locked
            print(f'Path {model_filename} is locked')
            # model_retrain[model_name].load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
            df_train = None
        elif not os.path.isfile(model_filename + '.lock'):
            print(f"Training resnet {model_filename}")
            
            # either we do not need a file or it does not exist (or it's locked)
            touch(model_filename + '.lock') # we want to have a file let's lock it
        
            args = Params()
            args.do_polar = do_polar

            # get the weights of the network
            dataloaders = datasets_transforms(args)
            # get the architecture of the network
            # model_retrain = torch.hub.load('pytorch/vision:v0.10.0', model_name, pretrained=True)            
            if model_name == 'resnet50':
                model_retrain = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
            else:
                model_retrain = torchvision.models.resnet101(weights=torchvision.models.ResNet101_Weights.DEFAULT)
                                
            if args.do_polar: 
                model_retrain = make_padding_circular_again(model_retrain)

            if False:
                #  use warmstart from center model
                # model_filename_center = model_filename.replace('boxes', 'center')
                # if os.path.isfile(model_filename_center):
                #     model_retrain.load_state_dict(torch.load(model_filename_center, map_location=torch.device(device)))
                # model_filename_previous = model_filename.replace('2023-12-21_boxes', '2023-12-21')
                # 2023-12-20_resnet50_boxes_do_polar=True.
                model_filename_previous = f'{data_cache}/2023-12-20_{model_name}_do_polar=True'
                if os.path.isfile(model_filename_previous):
                    model_retrain.load_state_dict(torch.load(model_filename_previous, map_location=torch.device(device)))
                else:
                    print(f"Could not find {model_filename_previous}...")

            print("Re-training pretrained model...", model_filename)
            print(f"Traning {model_name}, image_size={args.image_size}")
            since = time.time()

            model_retrain, df_train = train_model(args, model_retrain, dataloaders=dataloaders)
            
            elapsed_time = time.time() - since
            print(f"Training completed in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")
            print(f"Saving...{model_filename}")
            torch.save(model_retrain.state_dict(), model_filename)
            
            df_train.to_json(json_filename, orient='index', indent=2)
            if device=='cuda': torch.cuda.empty_cache()

            print()     


        if not(df_train is None):
            import lmfit
            from lmfit import Parameters, Model


            x = df_train['total_image'].values
            y = df_train['avg_acc_val'].values

            # Define the function
            def model(x, acc_max, acc_min, x_50) :
                return acc_max - (acc_max-acc_min) * np.exp( - x / x_50)

            # Init the sigmoid model as an lmfit Model object 
            mod = Model(model)
            pars = Parameters()
            # Add the initial parameters guesses
            pars.add_many(('acc_max', np.max(y), True,  0.0, 1.),
                        ('x_50', np.max(x)/20, True, 1, np.max(x)),
                        ('acc_min', 1/1000, True,  0.0, 1.))
            # And fit using least-square minimization
            out = mod.fit(y, pars, x=x, nan_policy='omit', max_nfev = 3000)
            # Print the result
            result = f"fitted max accuracy= {out.best_values['acc_max']:.2f}, speed= {out.best_values['x_50']:.1f}"
            print(result)

            df_train_roll = df_train.rolling(window=5, min_periods=1, center=False).mean()
            ax = df_train_roll.plot(x='total_image', y='avg_acc', 
                                c=color, ls='dashed',
                                grid=True, ax=ax, label='TRAIN: ' + json_filename.strip(data_cache + '/').strip('.json'))    
            ax = df_train_roll.plot(x='total_image', y='avg_acc_val', 
                                c=color, 
                                grid=True, ax=ax, label='VAL: ' + json_filename.strip(data_cache + '/').strip('.json') + '-' + result)    
            if os.path.isfile(model_filename + '.lock'): os.remove(model_filename + '.lock')

ax.set_ylim(.15, .85)            
ax.set_ylim(.05, .95)            
ax.set_yscale('logit')


In [ ]:
args = Params()
# args.root = f'{DATAROOT}/Imagenet_redux'
args.batch_size_val = 200
dataloaders = datasets_transforms(args)
args

In [ ]:
%ls cached_data/*_resnet101_do_polar=True.pt

In [ ]:
model_filename = 'cached_data/2023-12-31_resnet101_do_polar=True.pt'
model_filename = 'cached_data/2024-01-03_resnet101_do_polar=True.pt'

# https://pytorch.org/tutorials/beginner/saving_loading_models.html

if True:
    with torch.no_grad():
        model_retrain = torchvision.models.resnet101(weights=torchvision.models.ResNet101_Weights.DEFAULT)
        model_retrain = make_padding_circular_again(model_retrain)
        model_retrain.load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
        model_retrain = model_retrain.to(device)
        # Remember that you must call model.eval() to set dropout and batch normalization layers to evaluation mode before running inference. Failing to do this will yield inconsistent inference results.
        model_retrain = model_retrain.eval()


In [ ]:
from tqdm import tqdm
if True:
    with torch.no_grad():
        acc_val = 0
        n_val = len(dataloaders['val'])
        for _, (images, labels) in enumerate(tqdm(dataloaders['val'])):
            images, labels = images.to(device), labels.to(device)

            outputs = model_retrain(images).squeeze(1)

            _, preds = torch.max(outputs.data, 1)
            acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

        avg_acc_val = acc_val / n_val
avg_acc_val

In [ ]:
from tqdm import tqdm
if True:
    with torch.no_grad():
        acc_val = 0
        n_val = len(dataloaders['val'])
        for _, (images, labels) in enumerate(tqdm(dataloaders['val'])):
            images, labels = images.to(device), labels.to(device)

            outputs = model_retrain(images)

            _, preds = torch.max(outputs.data, dim=1)
            acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

        avg_acc_val = acc_val / n_val
avg_acc_val

In [ ]:
from tqdm import tqdm
if True:
    with torch.no_grad():
        acc_val = 0
        n_val = len(dataloaders['val'])
        for _, (images, labels) in enumerate(tqdm(dataloaders['val'])):
            images, labels = images.to(device), labels.to(device)

            outputs = model_retrain(images)

            _, preds = torch.max(outputs.data, dim=1)
            acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

        avg_acc_val = acc_val / n_val
avg_acc_val

In [ ]:
from tqdm import tqdm
if True:
    with torch.no_grad():
        acc_val = 0
        n_val = len(dataloaders['val'])
        for _, (images, labels) in enumerate(tqdm(dataloaders['val'])):
            images, labels = images.to(device), labels.to(device)

            outputs = model_retrain(images)

            _, preds = torch.max(outputs.data, dim=1)
            acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

        avg_acc_val = acc_val / n_val
avg_acc_val

In [ ]:
from tqdm import tqdm
for factor in range(10):
    args = Params()
    # args.root = f'{DATAROOT}/Imagenet_redux'
    args.batch_size_val = 2 * 2**factor
    dataloaders = datasets_transforms(args)
    model_retrain = torchvision.models.resnet101()
    model_retrain = make_padding_circular_again(model_retrain)
    model_retrain.load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
    model_retrain = model_retrain.to(device)
    # Remember that you must call model.eval() to set dropout and batch normalization layers to evaluation mode before running inference. Failing to do this will yield inconsistent inference results.
    with torch.no_grad():
        model_retrain = model_retrain.eval()

        acc_val = 0
        n_val = len(dataloaders['val'])
        for _, (images, labels) in enumerate(tqdm(dataloaders['val'])):
            images, labels = images.to(device), labels.to(device)

            outputs = model_retrain(images)

            _, preds = torch.max(outputs.data, dim=1)
            acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

        avg_acc_val = acc_val / n_val
    print(f'{avg_acc_val:.3f}')

In [ ]:
from tqdm import tqdm
for _ in range(10):
    args = Params()
    # args.root = f'{DATAROOT}/Imagenet_redux'
    args.batch_size_val = 400
    dataloaders = datasets_transforms(args)
    model_retrain = torchvision.models.resnet101()
    model_retrain = make_padding_circular_again(model_retrain)
    model_retrain.load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
    model_retrain = model_retrain.to(device)
    # Remember that you must call model.eval() to set dropout and batch normalization layers to evaluation mode before running inference. Failing to do this will yield inconsistent inference results.
    with torch.no_grad():
        model_retrain = model_retrain.eval()

        acc_val = 0
        n_val = len(dataloaders['val'])
        for _, (images, labels) in enumerate(tqdm(dataloaders['val'])):
            images, labels = images.to(device), labels.to(device)

            outputs = model_retrain(images)

            _, preds = torch.max(outputs.data, dim=1)
            acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

        avg_acc_val = acc_val / n_val
    print(f'{avg_acc_val:.3f}')

In [ ]:
model_retrain(images).squeeze(1).shape

In [ ]:
model_retrain(images).shape

In [ ]:
_, preds = torch.max(outputs.data, dim=1)
preds

In [ ]:
_, preds = torch.max(outputs, dim=1)
preds

In [ ]:
preds == labels.data

In [ ]:
torch.mean((preds == labels.data)*1.).cpu().item()

In [ ]:
torch.mean((preds == labels.data)*1.).cpu().item()

In [ ]:
torch.mean((preds == labels.data)*1.).cpu().item()

# optimize meta-parameters

In [ ]:
# print(path_save)
# %ls -l {path}*
# %rm {path} + '.sqlite3'

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
study_name = datetag + '_optuna'

model_name = 'resnet101'
do_polar = True
model_filename = f'{data_cache}/{datetag}_{model_name}_{do_polar=}.pt'

scan_dicts= {
             'image_size' : [224],
            }

label_dicts= {
             'image_size' : 'image size',
            }

In [ ]:
model_filename

In [ ]:
%ls {model_filename}

In [ ]:
subplotpars_scan = SubplotParams(left=0.125, right=.95, bottom=0.25, top=.975)
max_threshold = .999
for key in scan_dicts:
    filename = f'{data_cache}/{study_name}_{key}.json'
    if not(os.path.isfile(filename)):
        print(50*'=')
        print('Scanning along', key, "=", label_dicts[key])
        print(50*'=')
        if os.path.isfile(filename):
            df_scan = pd.read_json(filename)
        else:
            measure_columns = [key, 'accuracy']
            df_scan = pd.DataFrame([], columns=measure_columns)
            i_loc = 0
            for i_value, value in enumerate(scan_dicts[key]):
                print('i_value', i_value + 1, ' /', len(scan_dicts[key]), key, '=', value)

                opt =  Params()
                opt.root = f'{DATAROOT}/Imagenet_boxes' # Directory containing images to perform the training
                opt.n_train_stop = 100000
                opt.num_epochs = 1
                # opt.beta2 = 1.e-7

                new_dict = asdict(opt)
                new_dict[key] = value
                new_opt = Params(**new_dict)
                
                def objective(trial):
                    new_opt.rs_min = trial.suggest_float('rs_min', -1, 1.)
                    new_opt.rs_max = trial.suggest_float('rs_max', -6, -4)
                    scale = 4
                    new_opt.momentum = trial.suggest_float('momentum', opt.momentum/scale, min(opt.momentum*scale, max_threshold), log=True)
                    scale = 10
                    new_opt.lr = trial.suggest_float('lr', opt.lr / scale, opt.lr * scale, log=True)
                    scale = 50
                    # new_opt.beta2 = trial.suggest_float('beta2', opt.beta2/scale, min(opt.beta2*scale, 1.), log=True)

                    # load legacy model
                    if model_name == 'resnet50':
                        model_retrain = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
                    else:
                        model_retrain = torchvision.models.resnet101(weights=torchvision.models.ResNet101_Weights.DEFAULT)
                                        
                    # modify padding in resnet to make it circular
                    model_retrain = make_padding_circular_again(model_retrain)
                    model_retrain = model_retrain.to(device)

                    # use warmstart from center model
                    model_retrain.load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
                
                    # train and get accuracy on the validation set
                    dataloaders = datasets_transforms(new_opt, verbose=False)
                    _, df_train = train_model(new_opt, model_retrain, dataloaders=dataloaders, verbose=False)

                    accuracy = df_train['avg_acc_val'].mean()
                    # print(f'{new_opt.lr=} {new_opt.momentum=} : {accuracy}')
                    return accuracy

                opt_tuna= dict(storage=f"sqlite:///{os.path.join(data_cache, study_name)}.sqlite3", direction='maximize', load_if_exists=True,study_name=f"{key} = {value}")

                # 3. Create a study object and optimize the objective function.
                study = optuna.create_study(**opt_tuna)
                study.optimize(objective, n_trials=150, n_jobs=1, show_progress_bar=True)
                print(50*'-.')
                print("Best params: ", study.best_params)
                print("Best value: ", study.best_value)
                print("Best Trial: ", study.best_trial)
                print("Trials: ", study.trials)
                print(50*'-.')
                df_scan.loc[i_loc] = {key:value, 'accuracy':study.best_value}
                i_loc += 1
            df_scan.to_json(filename, orient='index', indent=2)
        print(df_scan)
        print(50*'=')

        fig, ax = plt.subplots(figsize=(fig_width, fig_width/phi), subplotpars=subplotpars_scan)
        gp_scan = df_scan[[key, 'accuracy']].groupby([key])
        means = gp_scan.mean()
        errors = gp_scan.std()
        means.plot.bar(yerr=errors, ax=ax, capsize=4, rot=-60, legend=False, color='r', alpha=.5)
        
        ax.set_ylabel('Accuracy')
        ax.set_xlabel(key + ' = ' +label_dicts[key])
        #ax.set_xscale('log')

        ax.set_ylim(0, 1)
        #fig = ax.get_figure()
        # pos = ax.get_position()
        # print(pos)
        plt.show()